# Your first circuits in Qiskit
**Quantum Computing · Week 1 (Qubits and measurement) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/qc/w1d4.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

A quantum **circuit** is a recipe: qubits start in |0⟩, gates change them, measurements read them out.
Today you build one, draw it, and run it 1000 times on a simulator.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(1, 1)     # 1 qubit, plus 1 classical bit to store the result
qc.h(0)                       # Hadamard gate on qubit 0: |0> becomes an equal mix of 0 and 1
qc.measure(0, 0)              # measure qubit 0, write the result into classical bit 0

qc.draw("mpl", style="clifford")

## Run it
A **sampler** runs the circuit many times (**shots**) and counts the outcomes.
`StatevectorSampler` is an exact simulator that runs on this computer.

In [ ]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler(seed=42)          # seed: repeatable "randomness"
job = sampler.run([qc], shots=1000)
result = job.result()[0]                       # results for our (first and only) circuit
counts = result.data.c.get_counts()            # "c" is the name of the classical register
print(counts)

In [ ]:
from qiskit.visualization import plot_histogram

plot_histogram(counts, color="#7b5cfa", title="1000 shots of H, then measure", figsize=(5.2, 3.4))

Each shot is one random bit, so this tiny circuit is a true random number generator
(on real hardware; here the simulator uses a pseudo-random seed).

In [ ]:
bits = result.data.c.get_bitstrings()
print("first 40 shots:", "".join(bits[:40]))

## Other one-qubit recipes
`x` is the quantum NOT gate. Predict each result **before** you run the cell!

In [ ]:
recipes = {
    "measure only": [],
    "X": ["x"],
    "H": ["h"],
    "X then H": ["x", "h"],
    "H then H": ["h", "h"],
}
circuits = []
for gates in recipes.values():
    c = QuantumCircuit(1, 1)
    for g in gates:
        getattr(c, g)(0)          # same as c.x(0) or c.h(0)
    c.measure(0, 0)
    circuits.append(c)

results = sampler.run(circuits, shots=1000).result()
for name, res in zip(recipes, results):
    print(f"{name:>12}: {res.data.c.get_counts()}")

Did "H then H" surprise you? One H gives 50/50, so two should still be random... but the answer
is always 0. That is **interference**, the star of week 3.

## Two qubits, two coins
`measure_all` measures every qubit. Qiskit prints bitstrings with qubit 0 on the **right**: `q1 q0`.

In [ ]:
qc2 = QuantumCircuit(2)
qc2.x(0)          # qubit 0 becomes 1 for sure
qc2.h(1)          # qubit 1 becomes a fair coin
qc2.measure_all()
print(qc2.draw("text"))
print(sampler.run([qc2], shots=1000).result()[0].data.meas.get_counts())

Only `01` and `11` appear: the right-hand character (qubit 0) is always 1.

## Your turn: three qubits
Build a 3-qubit circuit where qubit 0 is a fair coin, qubit 1 is always 1 and qubit 2 is always 0.
Predict the two bitstrings you'll see, then run it.

In [ ]:
qc3 = QuantumCircuit(3)
# TODO: add gates here
qc3.measure_all()
print(sampler.run([qc3], shots=1000).result()[0].data.meas.get_counts())

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
qc3 = QuantumCircuit(3)
qc3.h(0)          # qubit 0: fair coin
qc3.x(1)          # qubit 1: always 1
                  # qubit 2: no gate, stays 0
qc3.measure_all()
print(qc3.draw("text"))
print(sampler.run([qc3], shots=1000).result()[0].data.meas.get_counts())
print("Read right to left: q0 is the coin, q1 = 1, q2 = 0, so only 010 and 011 appear.")